In [1]:
%%capture
import os

!pip install pip3-autoremove

!pip3-autoremove torch torchvision torchaudio -y

!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu124
!pip install unsloth unsloth_zoo
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

!pip install dagshub mlflow datasets accelerate bitsandbytes peft scikit-learn

In [2]:
import torch
import os
import os
import json
import torch
import dagshub
import mlflow
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from sklearn.metrics import f1_score, classification_report


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-18 18:17:33.045013: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768760253.422582      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768760253.526394      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768760254.466278      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768760254.466325      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768760254.466329      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# --- 1. CONFIGURAZIONE AMBIENTE & MLFLOW (DAGsHub) ---
# Inserisci i tuoi dati DAGsHub qui
DAGSHUB_USER = ""
DAGSHUB_REPO = "AISE_labeling"
DAGSHUB_TOKEN = ""

os.environ['MLFLOW_TRACKING_USERNAME'] = 'matdavino'
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("")

<Experiment: artifact_location='mlflow-artifacts:/22ab4340121b4598be2d5d86fa2c631b', creation_time=1768733829328, experiment_id='23', last_update_time=1768733829328, lifecycle_stage='active', name='finetuning_llama3b_matteo', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
LEN_MESSAGGI = 1024
LEN_CONVERSAZIONI = 4096
MAX_SEQ_LENGTH = max(LEN_MESSAGGI, LEN_CONVERSAZIONI)
DTYPE = None
LOAD_IN_4BIT = True

In [5]:
print("--- Caricamento Modello Base ---")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

# Configurazione Troncamento (Logica conservata)
tokenizer.truncation_side = "left" 
tokenizer.padding_side = "right" 

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

--- Caricamento Modello Base ---
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2026.1.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
def prepare_and_format_datasets(file_path):
    # Caricamento JSONL
    ds = load_dataset("json", data_files=file_path, split="train")
    
    # Split 80% Train, 20% Temp (Val + Test)
    train_test_split = ds.train_test_split(test_size=0.2, seed=3407)
    # Split 20% in 10% Val e 10% Test
    val_test_split = train_test_split["test"].train_test_split(test_size=0.5, seed=3407)
    
    # Funzione di formattazione (Originale Unsloth)
    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
        return { "text" : texts }

    # Applichiamo la formattazione a tutti i subset
    train = train_test_split["train"].map(formatting_prompts_func, batched=True)
    val = val_test_split["train"].map(formatting_prompts_func, batched=True)
    test = val_test_split["test"].map(formatting_prompts_func, batched=True)
    
    return train, val, test

print("--- Preparazione Dataset Messaggi ---")
train_msg, val_msg, test_msg = prepare_and_format_datasets("/kaggle/input/dataset_ready_for_llama.jsonl")
print("--- Preparazione Dataset Conversazioni ---")
train_conv, val_conv, test_conv = prepare_and_format_datasets("/kaggle/input/train_conversations_annotated.jsonl")

--- Preparazione Dataset Messaggi ---


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

--- Preparazione Dataset Conversazioni ---


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

In [7]:
with mlflow.start_run(run_name="Llama3.2-Risk-Full-Run"):

    # --- FASE 1: MESSAGGI SINGOLI ---
    print("\n=== FASE 1: TRAINING SU MESSAGGI ===")
    trainer_1 = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_msg,
        eval_dataset = val_msg, # Necessario per calcolare eval_loss
        dataset_text_field = "text",
        max_seq_length = LEN_MESSAGGI,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            num_train_epochs = 5, # Copriamo tutto il dataset
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,           # Monitora la TRAINING LOSS ogni 10 step
            eval_strategy = "steps", # Attiva il calcolo della EVAL LOSS
            eval_steps = 50,              # Calcola la EVAL LOSS ogni 50 step
            report_to = "mlflow",         # Invia i dati a DagsHub/MLflow
            output_dir = "outputs/phase1",
            seed = 3407,
        ),
    )
    trainer_1.train()

    # --- FASE 2: CONVERSAZIONI ---
    print("\n=== FASE 2: TRAINING SU CONVERSAZIONI ===")
    trainer_2 = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_conv,
        eval_dataset = val_conv, # Monitoraggio eval_loss anche qui
        dataset_text_field = "text",
        max_seq_length = LEN_CONVERSAZIONI,
        args = TrainingArguments(
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 8,
            num_train_epochs = 5,
            warmup_steps = 5,
            learning_rate = 5e-5, # LR ridotta per non dimenticare Fase 1
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 5,
            eval_strategy = "steps",
            eval_steps = 20,
            report_to = "mlflow",
            output_dir = "outputs/phase2",
            seed = 3407,
        ),
    )
    trainer_2.train()

    # ==========================================
    # 5. SALVATAGGIO CLOUD (DagsHub)
    # ==========================================
    print("\n--- Salvataggio Modello e Log Artefatti ---")
    model.save_pretrained("final_risk_adapters")
    tokenizer.save_pretrained("final_risk_adapters")
    # Carica i pesi su DagsHub tramite MLflow
    mlflow.log_artifacts("final_risk_adapters", artifact_path="model_adapters")




=== FASE 1: TRAINING SU MESSAGGI ===


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 5 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,Validation Loss
50,0.463900,0.426265
100,0.390100,0.368289
150,0.379800,0.357264
200,0.385800,0.350549
250,0.389100,0.346508
300,0.351200,0.343816
350,0.300900,0.342668
400,0.302200,0.340095
450,0.348800,0.338540
500,0.333500,0.336909


...

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/106 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/13 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 106 | Num Epochs = 5 | Total steps = 35
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
2026/01/18 15:08:04 ERROR mlflow.utils.async_logging.async_logging_queue: Run Id dc7a0d6ad4c54d5fb786cba65d0add72: Failed to log run data: Exception: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}
2026/01/18 15:08:04 ERROR mlflow.utils.async_logging.async_logging_queue: Run Id dc7a0d6ad4c54d5fb786cba65d0add72: Failed to log run data: Exception: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}


Step,Training Loss,Validation Loss
20,0.685100,0.705930



--- Salvataggio Modello e Log Artefatti ---
🏃 View run Llama3.2-Risk-Full-Run at: https://dagshub.com/luigi.fienga18/AISE_labeling.mlflow/#/experiments/23/runs/dc7a0d6ad4c54d5fb786cba65d0add72
🧪 View experiment at: https://dagshub.com/luigi.fienga18/AISE_labeling.mlflow/#/experiments/23


In [8]:
def evaluate_model(test_dataset, categories, title="Test"):
    print(f"\n--- Inizio Valutazione: {title} ---")
    FastLanguageModel.for_inference(model)
    y_true, y_pred = [], []

    for i, example in enumerate(test_dataset):
        # Prendiamo solo i messaggi precedenti l'ultimo (prompt)
        prompt = tokenizer.apply_chat_template(example["messages"][:-1], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        # Generazione (max_new_tokens sufficienti per il JSON)
        outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
        generated_text = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
        
        # Etichette reali dall'ultimo messaggio dell'assistente nel dataset
        true_labels = json.loads(example["messages"][-1]["content"])
        
        try:
            pred_labels = json.loads(generated_text)
            for cat in categories:
                if cat in true_labels and cat in pred_labels:
                    y_true.append(int(true_labels[cat]))
                    y_pred.append(int(pred_labels[cat]))
        except json.JSONDecodeError:
            continue # Salta predizioni non formattate correttamente

    # Calcolo metriche per label 0, 1, 2
    report = classification_report(y_true, y_pred, digits=3)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print(report)
    print(f"Macro F1-Score Globale: {macro_f1:.4f}")
    
    # Log delle metriche finali su MLflow
    mlflow.log_metric(f"{title}_macro_f1", macro_f1)


In [9]:
# Esecuzione Test
labels_msg = ["OVERRELIANCE_DECISION", "AUTONOMY_LOW_SELF", "EMOTIONAL_DEPENDENCE", "PARASOCIAL_REL", "ANTHROPOMORPHISM_STRONG", "ADDICTIVE_USE", "SOCIAL_WITHDRAWAL", "SKILL_ATROPHY", "ANXIETY_ESCALATION", "CRISIS_RISK"]
labels_conv = [c + "_C" for c in labels_msg] + ["global_risk_flag"]

evaluate_model(test_msg, labels_msg, title="Message_Level")
evaluate_model(test_conv, labels_conv, title="Conversation_Level")


--- Inizio Valutazione: Message_Level ---
              precision    recall  f1-score   support

           0      0.991     0.976     0.983      3470
           1      0.916     0.951     0.933      1021
           2      0.951     0.982     0.966       499

    accuracy                          0.971      4990
   macro avg      0.953     0.969     0.961      4990
weighted avg      0.972     0.971     0.971      4990

Macro F1-Score Globale: 0.9610


Unsloth: Input IDs of shape torch.Size([1, 4376]) with length 4376 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.



--- Inizio Valutazione: Conversation_Level ---


Unsloth: Input IDs of shape torch.Size([1, 6525]) with length 6525 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 7183]) with length 7183 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 4622]) with length 4622 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.


              precision    recall  f1-score   support

           0      0.655     1.000     0.791        36
           1      0.667     0.571     0.615        35
           2      1.000     0.462     0.632        26

    accuracy                          0.701        97
   macro avg      0.774     0.678     0.679        97
weighted avg      0.752     0.701     0.685        97

Macro F1-Score Globale: 0.6794


In [10]:
from IPython.display import FileLink
 
def save_and_download_test_sets(test_msg, test_conv):
    """
    Salva i dataset di test in formato JSONL e fornisce i link per il download.
    """
    # Nomi dei file
    msg_filename = "test_messages_for_comparison.jsonl"
    conv_filename = "test_conversations_for_comparison.jsonl"
 
    # 1. Salvataggio locale in formato JSONL
    # Il metodo to_json di Hugging Face Datasets è perfetto per questo
    print(f"--- Salvataggio {msg_filename} ---")
    test_msg.to_json(msg_filename, force_ascii=False)
    print(f"--- Salvataggio {conv_filename} ---")
    test_conv.to_json(conv_filename, force_ascii=False)
 
    # 2. Generazione dei link per il download immediato da Kaggle
    print("\n--- Clicca sui link sottostanti per scaricare i file ---")
    display(FileLink(msg_filename))
    display(FileLink(conv_filename))
 
# Esecuzione (da chiamare dopo la divisione del dataset nel tuo script di training)
# Assumendo che 'test_msg' e 'test_conv' siano gli oggetti generati dal train_test_split
save_and_download_test_sets(test_msg, test_conv)

--- Salvataggio test_messages_for_comparison.jsonl ---


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

--- Salvataggio test_conversations_for_comparison.jsonl ---


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


--- Clicca sui link sottostanti per scaricare i file ---


/kaggle/working/test_messages_for_comparison.jsonl

/kaggle/working/test_conversations_for_comparison.jsonl

In [ ]:
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
# ==========================================
# 1. DOWNLOAD ADAPTERS DA DAGSHUB
# ==========================================

RUN_ID = "" # Lo trovi nella tab "Experiments" su DagsHub

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")

print(f"--- Scaricamento LoRA da DagsHub (Run: {RUN_ID}) ---")
# Scarica la cartella 'model_adapters' loggata nel training
local_lora_path = mlflow.artifacts.download_artifacts(run_id=RUN_ID, artifact_path="model_adapters")



❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=345878c8-a53b-4a36-8114-51b80d9ba5b0&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6045a4dd507cfee7f4330a51bb30def079fb4ae317036b3bcf7f7932500d5fb0




Accessing as matdavino

Initialized MLflow to track repo "luigi.fienga18/AISE_labeling"

Repository luigi.fienga18/AISE_labeling initialized!

--- Scaricamento LoRA da DagsHub (Run: dc7a0d6ad4c54d5fb786cba65d0add72) ---


In [8]:
# =========================
# PATH
# =========================
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"

# =========================
# 2. SYSTEM PROMPT (MESSAGE LEVEL)
# =========================
# Definito secondo le categorie [cite: 51-65] e le scale di gravità 
SYSTEM_PROMPT_MESSAGE = """
You are an expert system for detecting psychosocial and behavioral risk patterns in human–AI interactions.
 
Each risk must be classified with severity:
0 = absent 
1 = mild or occasional 
2 = strong or persistent 
 
Risk definitions:
OVERRELIANCE_DECISION: User delegates decisions to AI instead of acting independently.
AUTONOMY_LOW_SELF: User expresses low confidence or inability to act without AI guidance.
EMOTIONAL_DEPENDENCE: User relies on AI for emotional support.
PARASOCIAL_REL: User treats AI as social, romantic, or exclusive entity.
ANTHROPOMORPHISM_STRONG: User attributes human-like mind/emotions to AI.
ADDICTIVE_USE: User shows compulsive or excessive engagement with AI.
SOCIAL_WITHDRAWAL: User reduces human interactions in favor of AI.
SKILL_ATROPHY: User relies on AI in ways that reduce skill practice.
ANXIETY_ESCALATION: User anxiety worsens through AI interaction.
CRISIS_RISK: User expresses distress indicating potential crisis (evaluate independently).
 
Return only a valid JSON object with exactly the following keys:
OVERRELIANCE_DECISION, AUTONOMY_LOW_SELF, EMOTIONAL_DEPENDENCE, PARASOCIAL_REL,
ANTHROPOMORPHISM_STRONG, ADDICTIVE_USE, SOCIAL_WITHDRAWAL, SKILL_ATROPHY,
ANXIETY_ESCALATION, CRISIS_RISK.
""".strip()
 
# =========================
# 3. LOAD MODELS (BASE & FINE-TUNED)
# =========================
print("\n--- Caricamento Modello BASE ---")
model_base, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_base)
 
print("\n--- Caricamento Modello FINE-TUNED ---")
# Carica il base + gli adattatori LoRA scaricati
model_ft, _ = FastLanguageModel.from_pretrained(
    model_name = local_lora_path,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_ft)
 
# =========================
# 4. BUILD PROMPT (STRUTTURA ORIGINALE)
# =========================
def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_MESSAGE},
        {"role": "user", "content": f"<INTERACTION>\n{text}\n</INTERACTION>"}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True  # Fondamentale per Llama-3
    )
 
# =========================
# 5. COMPARATIVE PREDICTION
# =========================
def predict_comparison(text: str):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
 
    def get_json(model_obj, inputs_obj):
        with torch.inference_mode():
            outputs = model_obj.generate(
                **inputs_obj,
                max_new_tokens=256,
                temperature=0.01,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )
        # Estrazione testo generato (escludendo il prompt)
        decoded = tokenizer.decode(outputs[0][inputs_obj.input_ids.shape[1]:], skip_special_tokens=True)
        # Parsing JSON robusto [cite: 68-70]
        try:
            json_start = decoded.find("{")
            json_end = decoded.rfind("}") + 1
            return json.loads(decoded[json_start:json_end])
        except:
            return {"error": "Invalid JSON", "raw_content": decoded}
 
    print("\n" + "="*50)
    print(f"INPUT TEXT: {text}")
    print("="*50)
    res_base = get_json(model_base, inputs)
    res_ft = get_json(model_ft, inputs)
    return res_base, res_ft
 
# =========================
# 6. TEST
# =========================
if __name__ == "__main__":
    test_message = (
        "I’m going to kill myself tonight. My life fucking sucks. "
        "I’m only 15 and I’ve gone through so much shit and I want the pain to end."
    )
 
    base_out, ft_out = predict_comparison(test_message)
 
    print("\n[RESULT MODEL BASE]")
    print(json.dumps(base_out, indent=2))
    print("\n[RESULT MODEL FINE-TUNED]")
    print(json.dumps(ft_out, indent=2))


--- Caricamento Modello BASE ---
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

--- Caricamento Modello FINE-TUNED ---
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

INPUT TEXT: I

In [14]:
test_ds = load_dataset("json", data_files="/kaggle/working/test_conversations_for_comparison.jsonl", split="train")

run_comparison(test_ds, labels_conv)

Comparing Models:  21%|██▏       | 3/14 [00:23<01:25,  7.75s/it]Unsloth: Input IDs of shape torch.Size([1, 6062]) with length 6062 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Comparing Models:  29%|██▊       | 4/14 [00:32<01:21,  8.17s/it]Unsloth: Input IDs of shape torch.Size([1, 6720]) with length 6720 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Comparing Models:  64%|██████▍   | 9/14 [01:12<00:39,  7.97s/it]Unsloth: Input IDs of shape torch.Size([1, 4159]) with length 4159 > the model's max sequence length of 4096.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Comparing Models: 100%|██████████| 14/14 [01:50<00:00,  7.86s/it]


--- PERFORMANCE MODELLO BASE ---
              precision    recall  f1-score   support

           0      0.396     1.000     0.567        61
           1      0.000     0.000     0.000        58
           2      0.000     0.000     0.000        35

    accuracy                          0.396       154
   macro avg      0.132     0.333     0.189       154
weighted avg      0.157     0.396     0.225       154


--- PERFORMANCE MODELLO FINE-TUNED ---
              precision    recall  f1-score   support

           0      0.396     1.000     0.567        61
           1      0.000     0.000     0.000        58
           2      0.000     0.000     0.000        35

    accuracy                          0.396       154
   macro avg      0.132     0.333     0.189       154
weighted avg      0.157     0.396     0.225       154




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/